# Where Does Forgetting Live? (MLTA 2026)

Put the project folder in Google Drive as `MyDrive/mlta-forgetting`, pick a GPU (Runtime > Change runtime type > T4), then run the cells in order.

Results are saved on Drive. Turn the runtime off when a run is finished, because an idle session still uses compute units.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/mlta-forgetting
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Quick test (a few minutes)
Small model, tiny data. This only checks that everything runs.

In [ ]:
!python experiments.py run base lora --quick --model Qwen/Qwen2.5-0.5B-Instruct

## 1. Base model scores

In [ ]:
!python experiments.py run base

## 2. Naive fine-tuning
Run **one** of the two. `naive` is full fine-tuning and needs an A100. `naive_lora` works on a T4.

In [ ]:
!python experiments.py run naive_lora
# !python experiments.py run naive

Checkpoint: the general score of the naive model should be clearly below the base model. If it is not, stop and make the naive run more aggressive.

In [ ]:
!python pareto.py

## 3. Mitigations

In [ ]:
!python experiments.py run lora lora_r64 lora_lowlr replay20

## 4. Where does forgetting happen?

In [ ]:
!python experiments.py run attn_only mlp_only lower_half upper_half

In [ ]:
!python layer_analysis.py --name naive_lora --run runs/naive_lora
!python layer_analysis.py --name lora --run runs/lora

In [ ]:
from IPython.display import Image, display
display(Image('results/analysis/naive_lora_weights.png'), Image('results/analysis/naive_lora_drift.png'))

## 5. Model merging (no training)

In [ ]:
!python experiments.py merge naive_lora 0.25 0.5 0.75

## 6. RAG

In [ ]:
!python rag.py build
!python rag.py eval --name base
!python rag.py eval --name lora --run runs/lora

## 7. Summary table and trade-off plot

In [ ]:
!python pareto.py
from IPython.display import Image
Image('results/pareto.png')

## 8. Demo

In [ ]:
!python demo.py lora=runs/lora replay20=runs/replay20 --share